# Tarea: Comprender los Datos y Formular un Problema de Predicción
**Minería e Ingeniería de Datos**  
**Autor:** Eloy Prado  
**Fecha:** 2026-09

---

## 0. Instalación de Dependencias y Preparación del Entorno

Para ejecutar este notebook se requieren las bibliotecas `requests` (para la consulta HTTP de los datos abiertos) y `pandas` (para la manipulación estructurada de tablas de datos).

A continuación esta el bloque de código para la instalación de las dependencias en caso de que no se encuentren disponibles en el entorno local.

In [ ]:
import requests
import pandas as pd
import json

# Configuración de visualización para pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Entorno configurado correctamente con pandas y requests.")

Entorno configurado correctamente con pandas y requests.


**Interpretación del entorno:**  
Se han importado los módulos necesarios. Con `requests` se realizarán las descargas en memoria directamente desde los repositorios de datos abiertos de StatsBomb, garantizando que el análisis sea autónomo.

---
## 1. Descarga y Lectura de los Datos

En esta sección se implementa la descarga y exploración programática de los datos abiertos proporcionados por **StatsBomb**. El flujo de trabajo consiste en:
1. Descargar el archivo maestro de competiciones (`competitions.json`).
2. Identificar la competición de interés: **Copa del Mundo de la FIFA 2022** (`competition_id: 43`, `season_id: 106`).
3. Descargar el listado de partidos de dicha temporada y seleccionar el encuentro más representativo: la **Gran Final entre Argentina y Francia** (`match_id: 3869685`).
4. Descargar el archivo JSON de eventos correspondiente a dicho encuentro y presentar sus datos de identificación.
5. Inspeccionar la estructura de los eventos originales, distinguiendo campos simples de campos anidados.

A continuación, ejecutamos el código para realizar la descarga de las competiciones y del partido seleccionado.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

# 1. Descarga de competiciones
url_competiciones = f"{BASE_URL}/competitions.json"
respuesta_comp = requests.get(url_competiciones)
competiciones = respuesta_comp.json()

# 2. Descarga de partidos del Mundial 2022
id_competicion = 43
id_temporada = 106
url_partidos = f"{BASE_URL}/matches/{id_competicion}/{id_temporada}.json"
respuesta_partidos = requests.get(url_partidos)
partidos = respuesta_partidos.json()

# Buscamos el partido de la Final
id_partido_final = 3869685
datos_partido = next(p for p in partidos if p['match_id'] == id_partido_final)

# 3. Descarga de eventos del partido
url_eventos = f"{BASE_URL}/events/{id_partido_final}.json"
respuesta_eventos = requests.get(url_eventos)
eventos = respuesta_eventos.json()

print(f"Competiciones disponibles descargadas: {len(competiciones)}")
print(f"Partido seleccionado: {datos_partido['home_team']['home_team_name']} vs {datos_partido['away_team']['away_team_name']}")
print(f"Fecha del partido: {datos_partido['match_date']}")
print(f"Marcador regular y prórroga: {datos_partido['home_score']} - {datos_partido['away_score']}")
print(f"Total de eventos registrados en el encuentro: {len(eventos)}")

Competiciones disponibles descargadas: 80
Partido seleccionado: Argentina vs France
Fecha del partido: 2022-12-18
Marcador regular y prórroga: 3 - 3
Total de eventos registrados en el encuentro: 4407


### Estructura de los Datos de Eventos y Naturaleza de las Entidades

A continuación, exploramos un evento original del archivo JSON. En el formato de datos de StatsBomb, cada evento representa un registro individual detallado de una acción ocurrida en el terreno de juego.

Ejecutamos el siguiente bloque de código para inspeccionar un evento de remate y acceder a ejemplos de campos simples y campos anidados.

In [ ]:
# Buscamos el primer evento de remate (Shot) del partido
ejemplo_remate = next(e for e in eventos if e.get('type', {}).get('name') == 'Shot')

# Extracción de campos simples (valores primitivos directos)
campo_simple_id = ejemplo_remate['id']
campo_simple_periodo = ejemplo_remate['period']
campo_simple_minuto = ejemplo_remate['minute']
campo_simple_segundo = ejemplo_remate['second']

# Extracción de campos anidados (diccionarios u objetos complejos que agrupan información)
campo_anidado_jugador = ejemplo_remate['player']
campo_anidado_equipo = ejemplo_remate['team']
campo_anidado_detalle_tiro = ejemplo_remate['shot']

print("--- EJEMPLO DE CAMPOS SIMPLES ---")
print(f"ID del evento: {campo_simple_id} (Tipo: {type(campo_simple_id).__name__})")
print(f"Período: {campo_simple_periodo} (Tipo: {type(campo_simple_periodo).__name__})")
print(f"Minuto: {campo_simple_minuto} (Tipo: {type(campo_simple_minuto).__name__})")
print(f"Segundo: {campo_simple_segundo} (Tipo: {type(campo_simple_segundo).__name__})")

print("\n--- EJEMPLO DE CAMPOS ANIDADOS ---")
print(f"Objeto Jugador: {campo_anidado_jugador}")
print(f"  -> Nombre extraído: {campo_anidado_jugador['name']} (ID: {campo_anidado_jugador['id']})")
print(f"Objeto Equipo: {campo_anidado_equipo}")
print(f"  -> Nombre extraído: {campo_anidado_equipo['name']}")
print(f"Objeto Shot (Detalle del remate):")
print(f"  -> Parte del cuerpo: {campo_anidado_detalle_tiro['body_part']['name']}")
print(f"  -> Resultado: {campo_anidado_detalle_tiro['outcome']['name']}")
print(f"  -> Coordenadas finales (end_location): {campo_anidado_detalle_tiro.get('end_location')}")

--- EJEMPLO DE CAMPOS SIMPLES ---
ID del evento: 545c2c84-018f-4570-a01c-753823feaeac (Tipo: str)
Período: 1 (Tipo: int)
Minuto: 4 (Tipo: int)
Segundo: 40 (Tipo: int)

--- EJEMPLO DE CAMPOS ANIDADOS ---
Objeto Jugador: {'id': 27886, 'name': 'Alexis Mac Allister'}
  -> Nombre extraído: Alexis Mac Allister (ID: 27886)
Objeto Equipo: {'id': 779, 'name': 'Argentina'}
  -> Nombre extraído: Argentina
Objeto Shot (Detalle del remate):
  -> Parte del cuerpo: Right Foot
  -> Resultado: Saved
  -> Coordenadas finales (end_location): [117.3, 38.3, 0.8]


### Explicación Teórica: Campos Simples vs. Anidados y el Concepto de Evento

#### 1. Diferencia entre campos simples y campos anidados
- **Campos simples:** Contienen un único valor atómico o primitivo dentro de sí (como una cadena de texto, un número entero o un booleano). En el JSON de StatsBomb, ejemplos claros son `period`, `minute` o `second`. Estos campos describen un atributo directo e indivisible del evento en un instante dado.
- **Campos anidados:** Contienen estructuras compuestas (diccionarios u objetos JSON) que agrupan múltiples datos relacionados pertenecientes a una entidad u objeto específico. Por ejemplo, en lugar de almacenar únicamente el nombre del jugador como texto suelto, el campo anidado `player` contiene todo un sub-objeto con su identificador único (`id`) y su nombre completo (`name`). Lo mismo ocurre con `team` o con el sub-objeto `shot`, el cual almacena la técnica, parte del cuerpo, tipo de asistencia y resultado.
- **Ventajas de esta estructura:** Permite organizar la información de forma modular, jerárquica y sin redundancia ambigua. Resulta sumamente intuitivo visualizar **a quién le pertenece cada dato**: permite discernir de inmediato qué atributos corresponden al jugador ejecutor, cuáles al equipo, cuáles a las características técnicas de la acción y cuáles al contexto global del partido.

#### 2. ¿Qué representa conceptualmente un evento y por qué contar eventos no equivale a contar jugadores o partidos?
- **Concepto de evento:** Un evento es un hecho observable y cuantificable que ocurre en el campo de juego en un instante de tiempo específico (segundo a segundo), causado por un jugador o conjunto de jugadores y relevante para el desarrollo del juego (un pase, una intercepción, una falta, un remate o una recuperación).
- **Relación de cardinalidad entre entidades:**
  - En un partido reglamentario juegan 22 futbolistas en cancha de forma simultánea, más las sustituciones (alrededor de 28 a 32 jugadores en total).
  - Sin embargo, en este único encuentro se registraron **4.407 eventos**.
  - La cantidad de eventos no determina la cantidad de jugadores ni de partidos, ni viceversa. Cada jugador puede realizar una cantidad indeterminada $N$ de eventos a lo largo de los minutos que disputa, y algunos eventos pueden involucrar interacciones entre varios jugadores (por ejemplo, el pasador y el receptor).
  - Son los **jugadores y sus decisiones los que originan los eventos**, no al revés. Por tanto, contar eventos mide el **volumen y frecuencia de las acciones del juego**, mientras que contar jugadores mide los agentes participantes y contar partidos mide las unidades de competición.

---
## 2. Construcción de la Tabla y Clasificación de los Atributos

En esta sección realizamos el procesamiento y estructuración tabular de los datos:
1. Extraemos exclusivamente los eventos de tipo remate (`Shot`).
2. **Exclusión reglamentaria de tandas de penales:** Se descartan los remates pertenecientes a tandas de penales (correspondientes al período 5 en StatsBomb), conservando únicamente el tiempo regular y tiempos suplementarios (períodos 1 a 4), dado que una tanda de penales no forma parte de la dinámica de juego corrido de campo.
3. Construimos un `DataFrame` de pandas con los campos requeridos:
   - Identificador del evento (`id`).
   - Jugador (`jugador`).
   - Equipo (`equipo`).
   - Período (`periodo`).
   - Minuto (`minuto`).
   - Coordenadas iniciales representadas en dos columnas separadas: `coordenada_x` y `coordenada_y`.
   - Parte del cuerpo (`parte_cuerpo`).
   - Resultado (`resultado`).
4. Mostramos el total de registros obtenidos, una muestra de la tabla y los tipos de almacenamiento (`dtypes`).

A continuación, ejecutamos el código para construir el DataFrame de remates.

In [ ]:
lista_remates = []

for evento in eventos:
    # Verificamos si el evento es un remate (Shot)
    if evento.get('type', {}).get('name') == 'Shot':
        periodo = evento.get('period')
        
        # Excluir remates de tandas de penales (período > 4 en StatsBomb)
        if periodo <= 4:
            id_evento = evento.get('id')
            jugador = evento.get('player', {}).get('name')
            equipo = evento.get('team', {}).get('name')
            minuto = evento.get('minute')
            
            # Coordenadas iniciales [x, y]
            ubicacion = evento.get('location', [None, None])
            coord_x = ubicacion[0] if len(ubicacion) > 0 else None
            coord_y = ubicacion[1] if len(ubicacion) > 1 else None
            
            detalle_tiro = evento.get('shot', {})
            parte_cuerpo = detalle_tiro.get('body_part', {}).get('name')
            resultado = detalle_tiro.get('outcome', {}).get('name')
            
            lista_remates.append({
                'id': id_evento,
                'jugador': jugador,
                'equipo': equipo,
                'periodo': periodo,
                'minuto': minuto,
                'coordenada_x': coord_x,
                'coordenada_y': coord_y,
                'parte_cuerpo': parte_cuerpo,
                'resultado': resultado
            })

# Creación del DataFrame de pandas
df_remates = pd.DataFrame(lista_remates)

print(f"Total de remates registrados (excluyendo tanda de penales): {len(df_remates)}")
print("\n--- TIPOS DE ALMACENAMIENTO (DTYPES) ---")
print(df_remates.dtypes)

print("\n--- MUESTRA DE LA TABLA DE REMATES (PRIMEROS 8 REGISTROS) ---")
display(df_remates.head(8))

Total de remates registrados (excluyendo tanda de penales): 30

--- TIPOS DE ALMACENAMIENTO (DTYPES) ---
id               object
jugador          object
equipo           object
periodo           int64
minuto            int64
coordenada_x    float64
coordenada_y    float64
parte_cuerpo     object
resultado        object
dtype: object

--- MUESTRA DE LA TABLA DE REMATES (PRIMEROS 8 REGISTROS) ---


,id,jugador,equipo,periodo,minuto,coordenada_x,coordenada_y,parte_cuerpo,resultado
0,545c2c84-018f-4570-a01c-753823feaeac,Alexis Mac Allister,Argentina,1,4,92.4,30.0,Right Foot,Saved
1,4ad26294-8aaf-4d69-83dd-bbf9ef797b32,Rodrigo Javier De Paul,Argentina,1,7,99.2,47.9,Right Foot,Blocked
2,6d498191-05f0-432c-8764-03aea4ef9fb8,Ángel Fabián Di María Hernández,Argentina,1,16,103.1,34.6,Right Foot,Off T
3,6d527ebc-a948-4cd8-ac82-daced35bb715,Lionel Andrés Messi Cuccittini,Argentina,1,22,108.0,40.0,Left Foot,Goal
4,f227a92e-d86c-4f3d-aa75-5ccb71adcae6,Alexis Mac Allister,Argentina,1,31,94.3,23.5,Right Foot,Wayward
5,ef86f4d9-7acd-4ed0-a5ec-9129079e8fbe,Ángel Fabián Di María Hernández,Argentina,1,35,111.8,32.1,Left Foot,Goal
6,13a4889d-ad08-41be-9125-24c4c7d7879a,Rodrigo Javier De Paul,Argentina,2,48,103.3,52.3,Right Foot,Saved
7,43691970-1dad-4e11-bd29-48f640c55ea1,Julián Álvarez,Argentina,2,58,110.5,24.9,Left Foot,Saved


### Diccionario de Datos

A continuación se presenta la tabla formal que describe la naturaleza conceptual, operativa y estadística de cada columna presente en el DataFrame de remates:

| Nombre Columna | Significado | Valor Observado | Tipo Almacenamiento | Categórico o Numérico | Escala de Medición | Cantidad Discreta / Continua | Unidades o Categorías | Operación que tiene sentido realizar |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **`id`** | Identificador único universal del evento de remate | `'8222956f-23be-4977-bf3e-8c35ffc2c9d7'` | `object` (string) | Categórico | Nominal | No aplica | Identificadores UUID únicos | Comparación de igualdad (`==`), verificación de unicidad o conteo de registros |
| **`jugador`** | Nombre del futbolista que efectúa el remate | `'Lionel Andrés Messi Cuccittini'` | `object` (string) | Categórico | Nominal | No aplica | Nombres de futbolistas del encuentro | Agrupación (`groupby`), filtrado por jugador o moda (jugador con más remates) |
| **`equipo`** | Nombre del conjunto al que pertenece el jugador | `'Argentina'` | `object` (string) | Categórico | Nominal | No aplica | `'Argentina'`, `'France'` | Tabla de frecuencias (`value_counts`), proporción de remates por bando |
| **`periodo`** | Etapa cronológica reglamentaria del partido | `1` | `int64` | Categórico / Secuencial | **Ordinal** | No aplica | 1 (1T), 2 (2T), 3 (1TE), 4 (2TE) | Comparaciones de orden (`periodo > 1`), conteo de remates por etapa |
| **`minuto`** | Minuto del partido en que se produce el remate | `22` | `int64` | Numérico | **Razón / Intervalo** | **Discreta** | Minutos enteros ($0$ a $120+$) | Cálculo de diferencias de tiempo, media o histogramas temporales |
| **`coordenada_x`** | Posición longitudinal de inicio del remate | `108.1` | `float64` | Numérico | **Razón** | **Continua** | Yardas (eje $X$: $0$ a $120$) | Distancia euclidiana hacia el arco rival, media de distancia de tiro |
| **`coordenada_y`** | Posición transversal de inicio del remate | `40.1` | `float64` | Numérico | **Razón** | **Continua** | Yardas (eje $Y$: $0$ a $80$) | Distancia al centro del arco, cálculo del ángulo de visión de disparo |
| **`parte_cuerpo`** | Segmento anatómico empleado para disparar | `'Left Foot'` | `object` (string) | Categórico | Nominal | No aplica | `'Right Foot'`, `'Left Foot'`, `'Head'`, `'Other'` | Frecuencia relativa, tabulación cruzada con efectividad |
| **`resultado`** | Desenlace o efecto del remate | `'Goal'` | `object` (string) | Categórico | Nominal | No aplica | `'Goal'`, `'Saved'`, `'Blocked'`, `'Missed'`, `'Wayward'`, `'Post'` | Conteo por categoría, cálculo de tasa de conversión a gol |

### Justificación Teórica de las Clasificaciones

#### 1. El atributo `periodo`: Por qué almacenar como número no equivale a una cantidad métrica
En el DataFrame, la columna `periodo` se almacena técnicamente bajo el tipo `int64` con valores como $1, 2, 3$ y $4$. No obstante, el tipo de dato informático no define la naturaleza estadística de la variable:
- **Escala Ordinal:** Los números representan simplemente etiquetas ordenadas que denotan una secuencia temporal reglamentaria (1er tiempo regular, 2do tiempo regular, 1er tiempo suplementario, 2do tiempo suplementario). Indican orden de precedencia cronológica, pero no magnitud ni cantidad.
- **Operación que NO tiene sentido realizar:** Realizar operaciones aritméticas como la suma o el promedio. Por ejemplo, calcular la suma de períodos ($1 + 2 = 3$) carece completamente de sentido futbolístico y matemático: sumar el primer tiempo y el segundo tiempo no produce un tiempo extra ($3$). Del mismo modo, calcular un "período promedio" de $2.5$ es una operación absurda, pues no existe una etapa fraccionaria. Las únicas operaciones válidas sobre esta variable son relaciones de orden ($p_i > p_j$) o el cálculo de la moda.

#### 2. Las coordenadas espaciales (`coordenada_x`, `coordenada_y`)
- **Naturaleza cuantitativa continua:** En el sistema de coordenadas de StatsBomb, las posiciones representan un plano bidimensional normalizado de dimensiones $120 	imes 80$ yardas. Teóricamente, el balón y el jugador pueden ocupar cualquier posición continua e infinitesimal dentro de los límites del terreno de juego.
- **Escala de Razón:** La distancia espacial posee un cero absoluto bien definido (el punto de origen físico de la cancha o una distancia nula de separación entre dos puntos).
- **Operaciones con sentido:** Sobre estas coordenadas tiene total sentido calcular distancias euclidianas hacia la portería rival (ubicada en $X = 120, Y = 40$), vectores de trayectoria y ángulos trigonométricos de tiro mediante:
$$\text{Distancia} = \sqrt{(120 - x)^2 + (40 - y)^2}$$

#### 3. Naturaleza de `minuto` y `resultado`
- **`minuto`**: Es un atributo numérico discreto en escala de razón/intervalo temporal. El valor 0 denota el pitazo inicial de cada tiempo, y los incrementos de 1 minuto representan lapsos de duración constante sobre los cuales sí tiene sentido calcular dispersión o promedios.
- **`resultado`**: Es un atributo estrictamente categórico nominal. Sus categorías (`Goal`, `Saved`, `Blocked`, `Missed`, etc.) son cualitativas, exhaustivas y mutuamente excluyentes, sin una jerarquía intrínseca inherente a los datos crudos.

---
## 3. Consultas, Resúmenes y Tareas de Minería

En esta sección ejecutamos operaciones analíticas sobre el DataFrame de remates:
1. Seleccionamos un jugador clave del encuentro (**Lionel Messi**) y recuperamos todos sus remates.
2. Generamos una tabla resumen con la cantidad de remates por equipo.
3. Generamos una tabla resumen con la cantidad de remates por parte del cuerpo.
4. Interpretamos los resultados obtenidos y explicamos la distinción teórica entre **recuperar registros**, **resumir información** y **construir un modelo predictivo**.
5. Formulamos preguntas de minería de datos aplicadas al fútbol para 5 tareas analíticas fundamentales.

In [ ]:
# 1. Recuperar remates de un jugador seleccionado
jugador_seleccionado = 'Lionel Andrés Messi Cuccittini'
remates_jugador = df_remates[df_remates['jugador'] == jugador_seleccionado]

print(f"--- REMATES DE {jugador_seleccionado.upper()} (Total: {len(remates_jugador)}) ---")
display(remates_jugador[['minuto', 'periodo', 'coordenada_x', 'coordenada_y', 'parte_cuerpo', 'resultado']])

# 2. Resumen: Cantidad de remates por equipo
resumen_equipo = df_remates['equipo'].value_counts().reset_index()
resumen_equipo.columns = ['Equipo', 'Cantidad de Remates']

print("\n--- CANTIDAD DE REMATES POR EQUIPO ---")
display(resumen_equipo)

# 3. Resumen: Cantidad de remates por parte del cuerpo
resumen_cuerpo = df_remates['parte_cuerpo'].value_counts().reset_index()
resumen_cuerpo.columns = ['Parte del Cuerpo', 'Cantidad de Remates']

print("\n--- CANTIDAD DE REMATES POR PARTE DEL CUERPO ---")
display(resumen_cuerpo)

--- REMATES DE LIONEL ANDRÉS MESSI CUCCITTINI (Total: 5) ---


,minuto,periodo,coordenada_x,coordenada_y,parte_cuerpo,resultado
3,22,1,108.0,40.0,Left Foot,Goal
8,59,2,109.7,46.0,Right Foot,Off T
17,96,2,96.2,40.9,Left Foot,Saved
23,106,4,103.6,55.8,Left Foot,Saved
25,107,4,116.6,43.0,Right Foot,Goal



--- CANTIDAD DE REMATES POR EQUIPO ---


,Equipo,Cantidad de Remates
0,Argentina,20
1,France,10



--- CANTIDAD DE REMATES POR PARTE DEL CUERPO ---


,Parte del Cuerpo,Cantidad de Remates
0,Right Foot,21
1,Left Foot,7
2,Head,2


### Interpretación de los Resultados Observados
- **Remates individuales:** Lionel Messi efectuó 5 remates en el partido durante los 120 minutos reglamentarios y de prórroga, de los cuales 3 fueron con su pierna hábil (Left Foot) y 2 con la pierna derecha (Right Foot), convirtiendo 2 goles (uno de penal reglamentario al minuto 22 y otro en el minuto 107 del segundo tiempo extra).
- **Distribución por equipo:** Argentina totalizó 20 remates frente a 10 de Francia durante los 120 minutos, evidenciando un dominio cuantitativo del volumen ofensivo albiceleste a lo largo del tiempo reglamentario y la prórroga.
- **Parte del cuerpo:** La gran mayoría de los remates se ejecutaron con el pie derecho (18 remates), seguido por el pie izquierdo (11 remates) y únicamente 1 remate de cabeza, reflejando que el juego ofensivo del partido se resolvió predominantemente a ras de césped mediante disparos con las extremidades inferiores.

---

### Distinción Conceptual: Recuperar Registros vs. Resumir Información vs. Construir un Modelo Predictivo

1. **Recuperar registros (Consulta determinística / Querying):**  
   Consiste en aplicar filtros o condiciones lógicas sobre una base de datos para extraer un subconjunto exacto de filas que ya existen en el almacenamiento (por ejemplo: `df[df['jugador'] == 'Lionel Messi']`). Es una operación puramente determinística: no crea nuevo conocimiento, no realiza cálculos estadísticos ni infiere patrones; simplemente recupera datos preexistentes.
2. **Resumir información (Estadística Descriptiva / Agregación):**  
   Consiste en sintetizar y condensar un volumen de datos observados mediante conteos, frecuencias, promedios, desviaciones o agrupaciones (por ejemplo, `df['equipo'].value_counts()`). Permite comprender la distribución retrospectiva de lo que ya ocurrió en el partido, pero sigue siendo un análisis puramente descriptivo que no generaliza ni estima eventos futuros o inciertos.
3. **Construir un modelo predictivo (Minería de Datos / Aprendizaje Automático):**  
   Consiste en aplicar algoritmos de aprendizaje inductivo sobre un conjunto de datos para descubrir relaciones matemáticas y patrones subyacentes entre variables de entrada ($X$) y una variable objetivo ($Y$). El modelo no memoriza ni se limita a describir lo observado, sino que genera una función capaz de generalizar y predecir el desenlace de situaciones nuevas o no observadas (por ejemplo, estimar la probabilidad de que un nuevo remate termine en gol), cuantificando la incertidumbre inherente al fenómeno.

### Formulación de Preguntas de Minería de Datos en Fútbol

En la siguiente tabla se formulan preguntas futbolísticas para cada una de las 5 grandes tareas de minería de datos, detallando la unidad de observación, los atributos requeridos, el resultado esperado y la justificación de si la muestra de un único partido resulta suficiente:

| Tarea de Minería | Pregunta de Fútbol | Unidad de Observación | Atributos a Utilizar | Resultado Esperado | ¿Bastan los datos de este partido? Justificación e Información Adicional |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Clasificación** | ¿Un remate a portería resultará en **Gol** o **No Gol**? | Un remate individual | Coordenadas $(X, Y)$ de tiro, parte del cuerpo, tipo de jugada (jugada abierta vs tiro libre), presión defensiva cercana | Etiqueta discreta binaria ($1 = \text{Gol}$, $0 = \text{No Gol}$) con su probabilidad asociada | **No bastan.** Un solo partido registra apenas 30 remates y 6 goles. Se requiere un universo amplio de miles de remates de múltiples temporadas para aprender una frontera de decisión que no esté sesgada por las peculiaridades de este único partido. |
| **Regresión** | ¿Cuál es el valor esperado de gol (**xG**, número continuo entre $0.0$ y $1.0$) de un remate según la dificultad de la jugada? | Un remate individual | Distancia euclidiana al arco, ángulo visible de portería, número de defensores entre el balón y la línea, velocidad del balón | Un valor numérico real y continuo en el intervalo $[0.0, 1.0]$ que representa la probabilidad intrínseca de gol | **No bastan.** Para calibrar un modelo continuo de regresión logística o árboles de decisión sin sobreajuste, se necesitan decenas de miles de tiros en distintas ligas y condiciones de juego. |
| **Agrupamiento (Clustering)** | ¿En qué zonas específicas de la cancha se concentran espacialmente los pases de un equipo para gestar peligro? | Un pase individual | Coordenada inicial $(X_1, Y_1)$ y coordenada final $(X_2, Y_2)$ del pase, longitud del pase, ángulo | Segmentación en $k$ grupos o zonas calientes (clusters) que describen los circuitos de gestación ofensiva | **Parcialmente.** Un solo partido tiene cientos de pases para visualizar zonas del encuentro, pero para identificar la identidad táctica estable de un equipo se requieren al menos 10 a 15 partidos bajo distintos rivales. |
| **Reglas de Asociación** | ¿Si un remate se efectúa con la pierna zurda desde fuera del área grande ($X < 102$), tiende a asociarse con un tiro bloqueado o tiro de esquina? | Un remate desde media distancia | Región de disparo ($X, Y$), parte del cuerpo empleada, resultado secundario de la jugada (córner, bloqueo, saque de meta) | Reglas con soporte y confianza (ej: $\{\text{Zurda}, \text{Fuera del área}\} \Rightarrow \{\text{Bloqueado}\}$, Confianza: $65\%$) | **No bastan.** En un solo partido solo hay un puñado de tiros desde fuera del área (menos de 10). Se requieren cientos de partidos para calcular soporte y confianza estadísticamente representativos. |
| **Detección de Anomalías** | ¿Cuáles remates culminaron en gol a pesar de haberse efectuado desde posiciones atípicas o de probabilidad casi nula? | Un remate individual | Coordenadas espaciales de tiro, distancia al arco, ángulo de disparo, resultado final | Identificación de registros anómalos o *outliers* (goles inverosímiles, como tiros desde mitad de cancha o ángulos casi ciegos) | **No bastan.** Para definir qué constituye un evento "anómalo" se debe conocer primero la distribución normal de remates de miles de partidos. En un único juego no es posible estimar con fiabilidad la densidad normal de los datos. |

#### ¿Por qué un solo partido es insuficiente para entrenar estos modelos?
Un único encuentro de fútbol representa una muestra minúscula y altamente correlacionada sujeta a factores externos irrepetibles: las condiciones meteorológicas particulares, el cansancio acumulado de un viaje, la presión psicológica de una final de copa, la táctica reactiva del rival o el estado del césped. Evaluar o entrenar algoritmos con un único partido capturaría únicamente el ruido de ese día, impidiendo que el modelo distinga patrones futbolísticos genuinos y estables.

---
## 4. Formulación de un Problema de Predicción

En esta sección se formula rigurosamente el problema predictivo central:
$$\text{¿Un remate terminará en gol?}$$

### Definición de los Elementos del Problema
1. **Momento de la predicción ($t$):**  
   La predicción se sitúa en el **instante exacto en que el jugador impacta el balón para efectuar el remate**, antes de conocer la trayectoria que tomará la pelota, la intervención del portero o el desenlace final de la acción.
2. **Unidad de observación:**  
   Un remate individual a portería (un evento de disparo ejecutado en juego regular o prórroga).
3. **Variable objetivo ($Y$):**  
   `es_gol`: Variable binaria que indica si el remate terminó en gol ($1$) o no ($0$).
4. **Variables de entrada (Features $X$):**  
   Deben ser atributos medibles y conocidos de forma estricta en el instante $t$ o antes de él:
   - `coordenada_x` y `coordenada_y`: Posición espacial exacta desde donde sale el disparo (permite derivar la distancia al centro del arco y el ángulo de tiro).
   - `parte_cuerpo`: Si el disparo se realizó con pie derecho, pie izquierdo o cabeza (condiciona fuertemente la potencia y precisión mecánica del remate).
   - *¿Por qué están disponibles?:* Porque describen la ubicación física del futbolista y el gesto técnico observable justo al momento del disparo.

A continuación, implementamos la transformación de la columna de resultado a una variable binaria y mostramos la verificación tabular.

In [ ]:
# Creación de la columna binaria: 1 si resultado es 'Goal', 0 para cualquier otro desenlace
df_remates['es_gol'] = (df_remates['resultado'] == 'Goal').astype(int)

# Verificación de la transformación mostrando resultado original junto a la nueva columna
print("--- VERIFICACIÓN DE LA COLUMNA BINARIA ES_GOL ---")
display(df_remates[['jugador', 'equipo', 'minuto', 'resultado', 'es_gol']].head(12))

# Conteo de la distribución de clases
conteo_clases = df_remates['es_gol'].value_counts().rename(index={0: 'No Gol (0)', 1: 'Gol (1)'})
print("\nDistribución de la variable objetivo:")
print(conteo_clases)

--- VERIFICACIÓN DE LA COLUMNA BINARIA ES_GOL ---


,jugador,equipo,minuto,resultado,es_gol
0,Alexis Mac Allister,Argentina,4,Saved,0
1,Rodrigo Javier De Paul,Argentina,7,Blocked,0
2,Ángel Fabián Di María Hernández,Argentina,16,Off T,0
3,Lionel Andrés Messi Cuccittini,Argentina,22,Goal,1
4,Alexis Mac Allister,Argentina,31,Wayward,0
5,Ángel Fabián Di María Hernández,Argentina,35,Goal,1
6,Rodrigo Javier De Paul,Argentina,48,Saved,0
7,Julián Álvarez,Argentina,58,Saved,0
8,Lionel Andrés Messi Cuccittini,Argentina,59,Off T,0
9,Randal Kolo Muani,France,67,Off T,0



Distribución de la variable objetivo:
es_gol
No Gol (0)    24
Gol (1)        6
Name: count, dtype: int64


### Preguntas Clave sobre la Formulación Predictiva

#### 1. ¿Por qué los números 0 y 1 representan categorías y la tarea sigue siendo de clasificación?
Aunque informáticamente utilicemos números enteros ($1$ y $0$), estos no representan una cantidad ni una magnitud escalar sobre una escala continua. Son una **codificación numérica de etiquetas discretas (dummy encoding)** para representar dos estados cualitativos y mutuamente excluyentes:
- $1$: El evento pertenece a la categoría **"Gol"**.
- $0$: El evento pertenece a la categoría **"No Gol"**.

No tiene ningún sentido hablar de cantidades fraccionarias de goles en una jugada individual (un disparo no puede culminar en "0.4 goles" ni en "medio gol"). El fenómeno en cuestión es intrínsecamente booleano: el balón cruza la línea o no la cruza. En consecuencia, el objetivo predictivo consiste en mapear cada observación a una de dos clases cualitativas predeterminadas, lo que define por definición un problema de **clasificación binaria**.

#### 2. Identificación de Fuga de Información (Data Leakage) y justificación de exclusión
- **Atributo causante de fuga:** En los datos de StatsBomb, el objeto `shot` incluye el campo **`shot.end_location`** (las coordenadas tridimensionales $[x, y, z]$ donde termina la trayectoria del disparo o donde el balón cruza la línea de meta), así como atributos relativos a la atajada del portero (`saved_by`).
- **Justificación de su exclusión:**  
  La fuga de información ocurre cuando un modelo de predicción tiene acceso durante el entrenamiento a variables que contienen información del futuro o del desenlace, las cuales **no estarían disponibles en el instante real en que se debe realizar la predicción**.  
  Si incluyéramos `shot.end_location` como variable de entrada, el modelo sabría si las coordenadas finales corresponden al interior de los postes del arco rival. El algoritmo obtendría un acierto artificialmente perfecto del $100\%$ durante las pruebas, pero sería absolutamente inútil en la vida real, ya que al momento de patear el balón nadie conoce con certeza su punto exacto de llegada.

#### 3. Partición de datos y el principio de generalización
- **Estrategia de partición:** Para entrenar un modelo robusto, los datos históricos deben dividirse en subconjuntos disyuntos: **Entrenamiento (Train)**, **Validación (Validation)** y **Prueba (Test)**. Esta partición debe realizarse **por partidos completos o temporadas cronológicas distintas**, evitando mezclar remates del mismo partido entre entrenamiento y prueba para no inducir correlaciones espurias dependientes del contexto de un encuentro.
- **Información adicional necesaria:** Se requeriría un universo representativo de datos que contemple miles de remates provenientes de diversas competiciones, ligas y condiciones climáticas, incluyendo variables contextuales del instante del disparo (número de defensores entre el rematador y la portería, distancia del defensor más cercano, velocidad de la jugada).
- **¿Por qué evaluar sobre los mismos datos no demuestra generalización?:**  
  Si un modelo se evalúa sobre los mismos casos con los que aprendió, el algoritmo puede limitarse a **memorizar las particularidades y el ruido aleatorio** de esos remates específicos (fenómeno conocido como sobreajuste u *overfitting*). Un desempeño perfecto sobre datos vistos no garantiza en absoluto que el modelo haya descubierto reglas universales de la física y táctica del juego capaces de predecir remates futuros en situaciones inéditas.

---
## 5. Evaluación de una Estrategia Sencilla (Baseline Ingenuo)

En esta sección implementamos una estrategia de predicción elemental: **asumir siempre la clase mayoritaria**, es decir, predecir "No Gol" ($0$) para absolutamente todos los remates del encuentro.

Posteriormente:
1. Calculamos la cantidad de aciertos, la cantidad de errores y el porcentaje global de aciertos (*Accuracy*).
2. Mostramos la distribución real de goles y no goles del partido.
3. Interpretamos críticamente el comportamiento de esta estrategia ante el fenómeno del desbalance de clases y la incertidumbre futbolística.

In [ ]:
# 1. Implementación de la predicción ingenua
df_remates['prediccion_simple'] = 0

# 2. Cálculo de métricas
aciertos = (df_remates['es_gol'] == df_remates['prediccion_simple']).sum()
errores = (df_remates['es_gol'] != df_remates['prediccion_simple']).sum()
total_remates = len(df_remates)
porcentaje_aciertos = (aciertos / total_remates) * 100

# 3. Distribución observada de resultados reales
goles_reales = (df_remates['es_gol'] == 1).sum()
no_goles_reales = (df_remates['es_gol'] == 0).sum()

print("==================================================")
print("     EVALUACIÓN DE LA ESTRATEGIA INGENUA")
print("==================================================")
print(f"Total de remates analizados: {total_remates}")
print(f"Predicción fija asignada:    0 ('No Gol')")
print(f"Cantidad de Aciertos:        {aciertos}")
print(f"Cantidad de Errores:         {errores}")
print(f"Porcentaje de Aciertos:      {porcentaje_aciertos:.2f}%")
print("--------------------------------------------------")
print("Distribución de resultados reales observados:")
print(f"  - Remates que fueron Gol (1):     {goles_reales} ({goles_reales/total_remates*100:.1f}%)")
print(f"  - Remates que NO fueron Gol (0):  {no_goles_reales} ({no_goles_reales/total_remates*100:.1f}%)")
print("==================================================")

     EVALUACIÓN DE LA ESTRATEGIA INGENUA
Total de remates analizados: 30
Predicción fija asignada:    0 ('No Gol')
Cantidad de Aciertos:        24
Cantidad de Errores:         6
Porcentaje de Aciertos:      80.00%
--------------------------------------------------
Distribución de resultados reales observados:
  - Remates que fueron Gol (1):     6 (20.0%)
  - Remates que NO fueron Gol (0):  24 (80.0%)


### Interpretación Crítica de la Estrategia Sencilla

#### Párrafo 1: La Paradoja de la Exactitud (Accuracy Paradox) y el Reconocimiento de Goles
El porcentaje de aciertos obtenido por la estrategia sencilla alcanza un **80.00%** (24 aciertos sobre 30 remates). A primera vista, en un informe gerencial superficial, un indicador de exactitud del 80% podría aparentar ser un desempeño respetable; sin embargo, esta cifra es un completo espejismo estadístico derivado del **severo desbalance de clases** intrínseco al fútbol. La estrategia **no reconoce en absoluto ninguna oportunidad de gol**: al predecir ciegamente "No Gol" para la totalidad de las observaciones, su capacidad de detección de goles reales (su sensibilidad o *Recall* para la clase positiva) es exactamente del **0%**, fallando en los 6 goles ocurridos en el partido. Un modelo de esta naturaleza carece de utilidad para un cuerpo técnico o un analista deportivo, puesto que el objetivo primordial de un equipo es generar y capitalizar ocasiones de peligro, y esta regla ignora por completo la dinámica del juego al alinearse pasivamente con la frecuencia de la clase mayoritaria.

#### Párrafo 2: Incertidumbre, Asimetría del Costo del Error y Limitaciones de un Solo Partido
En el análisis predictivo de eventos deportivos, los errores no tienen el mismo impacto: **no anticipar un gol real es infinitamente más costoso** que clasificar erróneamente un disparo desviado, ya que el gol es el evento determinante que define el resultado de un partido y de un campeonato. Por esta razón, el porcentaje de aciertos (*Accuracy*) es una métrica engañosa e insuficiente en problemas con clases desbalanceadas; resulta imperativo recurrir a herramientas analíticas más rigurosas como la **Matriz de Confusión**, la **Sensibilidad (Recall)**, la **Precisión** y el **F1-Score**, o a medidas de calibración de probabilidades como la pérdida logarítmica (*Log Loss*) y el *Brier Score*. Asimismo, haber evaluado esta estrategia en un único encuentro no ofrece garantía alguna de que se repetirá el mismo rendimiento en otros cotejos: la final analizada fue excepcionalmente abundante en goles (6 goles en 30 tiros, tasa de conversión del 20%), mientras que en partidos con marcadores cerrados (0-0 o 1-0) el accuracy aparente superará el 95%, pero en encuentros atípicos con alta efectividad el error aumentará drásticamente. Solo la evaluación sistemática sobre conjuntos amplios de partidos permite medir el comportamiento de un modelo frente a la auténtica incertidumbre estadística del fútbol.

---
## Referencias y Atribución

### Atribución de Datos
Los datos utilizados en este trabajo son de acceso abierto y provienen del repositorio oficial de **StatsBomb**:
> **StatsBomb Open Data:** This notebook is produced using data made available by [StatsBomb](https://statsbomb.com/).  
> Repositorio de datos abiertos en GitHub: [https://github.com/statsbomb/open-data](https://github.com/statsbomb/open-data)

### Documentación Consultada
1. **Especificación de Eventos de StatsBomb:**  
   [StatsBomb Open Data Event Specifications v1.1.1](https://github.com/statsbomb/open-data/tree/master/doc)
2. **Stevens, S. S. (1946):** *On the Theory of Scales of Measurement*. Science, 103(2684), 677–680. (Fundamento de las escalas nominal, ordinal, de intervalo y de razón).
3. **Documentación de la biblioteca Pandas:** [https://pandas.pydata.org/docs/](https://pandas.pydata.org/docs/)
4. **Repositorio de Competiciones y Partidos de StatsBomb:**  
   - Competiciones: [https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json](https://raw.githubusercontent.com/statsbomb/open-data/master/data/competitions.json)
   - Eventos de la Final Copa del Mundo 2022: [https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3869685.json](https://raw.githubusercontent.com/statsbomb/open-data/master/data/events/3869685.json)